In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# S05 — Design the evaluation before building the predictor

Read the matching lesson before running. Predictions first; source facts, manufactured controls and interpretations are labeled separately. This is not a sports performance benchmark.

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
from dataclasses import asdict
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/shape_lab').exists())
sys.path.insert(0, str(ROOT / 'src'))
from shape_lab import sports_v8 as sport
from shape_lab.persistence import finite_bottleneck
spl, profiles = sport.load_sports(ROOT)
frames = np.array([r['frame'] for r in spl['records']])
times = sport.frame_times(frames, spl['sampling_rate'])
ball = sport.to_metres([r['ball'] for r in spl['records']], spl['xyz_unit'])
report_dir = ROOT / 'reports/v8/sports'
report_dir.mkdir(parents=True, exist_ok=True)
def save_report(stage, result):
    # Reports contain ordinary finite values; absent values are recorded explicitly.
    (report_dir / f'{stage}.json').write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Data:', len(frames), 'selected frames from one shot;', len(profiles), 'profiles from', profiles.player_id.nunique(), 'players')


Data: 12 selected frames from one shot; 12 profiles from 8 players


## Partition the observed athlete IDs, not individual profile rows

In [3]:
groups=profiles.player_id.to_numpy()
parts=sport.group_partition(groups,[211,218,2759,2858],[4322,4792],[5468,5521])
for name,indices in parts.items():print(name,indices.tolist(),sorted(set(groups[indices])))
assert not set(groups[parts['train']]) & set(groups[parts['test']])
assert sum(map(len,parts.values()))==len(profiles)


train [0, 1, 2, 3, 4, 5] [np.int64(211), np.int64(218), np.int64(2759), np.int64(2858)]
validation [6, 7, 8] [np.int64(4322), np.int64(4792)]
test [9, 10, 11] [np.int64(5468), np.int64(5521)]


## Preprocessing is fitted on training data

In [4]:
features=['running_distance_full_all','hsr_distance_full_all','sprint_distance_full_all']
x=profiles[features].to_numpy()
z,train_mean,scale=sport.fit_standardize(x,parts['train'])
all_mean=x.mean(axis=0)
display(pd.DataFrame({'feature':features,'train_mean':train_mean,'all_row_mean_leakage_control':all_mean,'difference':all_mean-train_mean}))
assert np.allclose(z[parts['train']].mean(axis=0),0)


,feature,train_mean,all_row_mean_leakage_control,difference
0,running_distance_full_all,1311.603333,1301.735833,-9.8675
1,hsr_distance_full_all,426.848333,441.120833,14.2725
2,sprint_distance_full_all,90.110000,97.092500,6.9825


## Record what this exercise did NOT establish

In [5]:
result={'partition_rows':{k:v.tolist() for k,v in parts.items()},'partition_players':{k:sorted(map(int,set(groups[v]))) for k,v in parts.items()},'training_mean':train_mean.tolist(),'all_mean':all_mean.tolist(),'model_fitted':False,'sports_predictive_accuracy':None,'reason':'Small ordered excerpt; no independent supervised performance claim.'}
save_report('S05',result)
print(json.dumps(result,indent=2))


{
  "partition_rows": {
    "train": [
      0,
      1,
      2,
      3,
      4,
      5
    ],
    "validation": [
      6,
      7,
      8
    ],
    "test": [
      9,
      10,
      11
    ]
  },
  "partition_players": {
    "train": [
      211,
      218,
      2759,
      2858
    ],
    "validation": [
      4322,
      4792
    ],
    "test": [
      5468,
      5521
    ]
  },
  "training_mean": [
    1311.6033333333332,
    426.84833333333336,
    90.11000000000001
  ],
  "all_mean": [
    1301.7358333333334,
    441.12083333333334,
    97.09250000000002
  ],
  "model_fitted": false,
  "sports_predictive_accuracy": null,
  "reason": "Small ordered excerpt; no independent supervised performance claim."
}


## Independent explanation
State the observation unit, assumptions, units, one result and one unsupported conclusion. Then work the separate learner notebook without the answer notebook open.